In [2]:
import numpy as np
import math
from datetime import datetime, timedelta
from dash import Dash
from src.data.data_tools import Database, SENSOR_LIST, ddmm_to_decimal, export_imu_readings_to_csv
from src.ekf.ekf_utils import ekf_navigation, ekf_to_coor

app = Dash()

sample_limit = 100
db = Database("./data/dataset.db")

gps_readings = db.get_gps_readings(start_datetime=datetime(2026, 3, 20, 1), end_datetime=datetime(2026, 3, 21), min_satellite_count=4)

# subsample list to defined count starting at a zero velocity sample
for idx, reading in enumerate(gps_readings):
    if reading.speed <= 0.05 and idx + 1 < len(gps_readings) and gps_readings[idx + 1].speed <= 0.05:
        gps_readings = gps_readings[idx:idx+sample_limit]
        break

for idx, reading in enumerate(gps_readings):
    if idx > 0 and reading.timestamp - gps_readings[idx-1].timestamp > timedelta(seconds=1.1):
        gps_readings = gps_readings[0:idx]
        break

imu_readings = db.get_imu_readings(start_datetime=gps_readings[0].timestamp, end_datetime=gps_readings[-1].timestamp)

gps_coor = [(ddmm_to_decimal(gps.latitude), ddmm_to_decimal(gps.longitude)) for gps in gps_readings]
start_lat, start_lon = gps_coor[0]

ekf_q_map = {}
ekf_p_map = {}
ekf_v_map = {}
ekf_coor_map = {}
min_sample = math.inf

export_imu_readings_to_csv(imu_readings, "imu_readings.csv")

for sensor in SENSOR_LIST:

    # decompose sensor readings to separate arrays
    sensor_readings = [r for r in imu_readings if r.sensor_name == sensor]
    gyr = np.array([np.array(r.gyr) for r in sensor_readings])
    acc = np.array([np.array(r.acc) for r in sensor_readings])
    mag = np.array([np.array(r.mag) for r in sensor_readings])
    time = np.array([r.timestamp for r in sensor_readings])

    print(gyr)
    print(f"Running EKF on sensor {sensor} - {len(sensor_readings)} readings")
    print(f"Initial GPS data:"
          f"\n\tposition: {start_lat:.2f}, {start_lon:.2f}"
          f"\n\tcourse: {gps_readings[0].course}"
          f"\n\tsamples: {len(gps_coor)}")
    print(f"Initial IMU data:"
          f"\n\tmag heading: {math.degrees(math.atan2(mag[0, 0], mag[0, 1]))}"
          f"\n\tsamples: {len(sensor_readings)}"
          f"\n\trate: {len(sensor_readings)/len(gps_coor)}")

    Q, V, P = ekf_navigation(gyr, acc, mag, time, mag_ref=[.237617,	.45396,	.383970])

    ekf_coor_map[sensor] = ekf_to_coor(start_lat, start_lon, P)
    min_sample = min(min_sample, len(ekf_coor_map[sensor]))

    ekf_q_map[sensor] = Q
    ekf_p_map[sensor] = P
    ekf_v_map[sensor] = V

stacked = np.array([ekf_p_map[s][:min_sample] for s in SENSOR_LIST])  # shape (6, min_sample, 3)
avg_ekf = stacked.mean(axis=0)

# ekf_coor_map["avg"] = ekf_to_coor(start_lat, start_lon, avg_ekf)

Successfully exported 62188 records to 'imu_readings.csv'.
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 ...
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Running EKF on sensor icm_20948_1 - 10365 readings
Initial GPS data:
	position: 34.17, -119.22
	course: 182.32
	samples: 100
Initial IMU data:
	mag heading: 0.0
	samples: 10365
	rate: 103.65
Bias = [0. 0. 0.]


ValueError: A-priori quaternion must have a norm equal to 1.

In [ ]:
"""
Plot position data
"""

from src.data.plotting import gps_scatter_map, imu_scatter_map
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc
import numpy as np


map_fig = go.Figure()

gps_scatter_map(map_fig, gps_readings)

# Add EKF Traces to Map
for sensor in ekf_coor_map.keys():
    imu_scatter_map(
        map_fig,
        gps_readings[0].latitude,
        gps_readings[0].longitude,
        imu_readings[0].sensor_name,
        [imu.timestamp for imu in imu_readings if imu.sensor_name == sensor],
        ekf_q_map[sensor],
        ekf_v_map[sensor],
        ekf_p_map[sensor]
    )

gps_lats, gps_lons, gps_times = zip(*[(ddmm_to_decimal(gps.latitude), ddmm_to_decimal(gps.longitude), (gps.timestamp - gps_readings[0].timestamp).total_seconds()) for gps in gps_readings])

center_lat = np.mean(gps_lats)
center_lon = np.mean(gps_lons)

# Center map on GPS data
map_fig.update_layout(
    map=dict(style='open-street-map', center=dict(lat=center_lat, lon=center_lon), zoom=15),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
)


# 3. DASH APP LAYOUT
app = Dash(__name__)

app.layout = html.Div([
    html.Div([
        dcc.Graph(figure=map_fig)
    ], style={'padding': '0px'}),
])

app.run()

In [ ]:
"""
Plot velocity
"""



# 2. CREATE THE VELOCITY LINE CHART FIGURE (The new part)
vel_fig = go.Figure()

axis_colors = {
    'x': 'royalblue',
    'y': 'crimson',
    'z': 'forestgreen'
}

for sensor, velocities in ekf_v_map.items():
    # We use an index as the X-axis (Time/Sample count)
    # If you have a timestamp array for each sensor, replace np.arange with that.
    v_arr = np.array(velocities)
    time_axis = [imu.timestamp for imu in imu_readings if
                 gps_readings[0].timestamp <= imu.timestamp <= gps_readings[-1].timestamp and (
                             imu.sensor_name == sensor)]
    # time_axis = np.arange(len(v_arr))
    # Extract components
    vx = v_arr[:, 0]
    vy = v_arr[:, 1]
    vz = v_arr[:, 2]

    # Add Trace for X-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vx,
        mode='lines',
        name=f'{sensor} (N)',
        line=dict(color=axis_colors['x'], width=1.5),
        opacity=0.8  # Slightly transparent to handle overlapping lines
    ))

    # Add Trace for Y-component
    vel_fig_trace_y = vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vy,
        mode='lines',
        name=f'{sensor} (E)',
        line=dict(color=axis_colors['y'], width=1.5),
        opacity=0.8
    ))

    # Add Trace for Z-component
    vel_fig.add_trace(go.Scatter(
        x=time_axis, y=vz,
        mode='lines',
        name=f'{sensor} (D)',
        line=dict(color=axis_colors['z'], width=1.5),
        opacity=0.8
    ))



vn = [gps.speed * np.cos(np.radians(gps.course) )for gps in gps_readings]
ne = [gps.speed * np.sin(np.radians(gps.course) )for gps in gps_readings]
time_axis = [gps.timestamp for gps in gps_readings]


vel_fig.add_trace(go.Scatter(
    x=time_axis, y=vn,
    mode='lines',
    name='GPS (N)',
    line=dict(color="purple", width=1.5),
    opacity=0.8
))

vel_fig.add_trace(go.Scatter(
    x=time_axis, y=ne,
    mode='lines',
    name='GPS (E)',
    line=dict(color="goldenrod", width=1.5),
    opacity=0.8
))

vel_fig.update_layout(
    title='Sensor Velocity Over Time',
    xaxis_title='Sample Index (Time)',
    yaxis_title='Velocity (m/s)',
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    margin=dict(l=40, r=20, t=50, b=40)
)


# 3. DASH APP LAYOUT
app = Dash(__name__)

app.layout = html.Div([
    # Container for the Velocity Chart
    html.Div([
        dcc.Graph(figure=vel_fig)
    ], style={'padding': '0px'})
])

app.run()